# M-κ Cross-Section Shape Verification

Verification notebook for `MKappaAnalysis` across the most relevant RC cross-section shapes:

1. **Rectangular section** — standard RC beam (b×h, single bottom reinforcement layer)
2. **T-section** — floor beam / slab strip (flange at top, web, bottom tension steel)

**Goal:** confirm robust solver convergence, correct force equilibrium, and physically
sensible M-κ curves for each shape type.

**Coordinate conventions used throughout:**
- `z` in `ReinforcementLayer` = distance from the *bottom* fibre (z=0 at bottom)
- `h_total` = total section height
- Standard sagging moment positive; bottom fibre in tension


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Core scite imports
from scite.cs_design.shapes import RectangularShape, TShape
from scite.cs_design.cross_section import CrossSection
from scite.cs_design.reinforcement import ReinforcementLayer, ReinforcementLayout
from scite.matmod.ec2_parabola_rectangle import EC2ParabolaRectangle
from scite.matmod.steel_reinforcement import SteelReinforcement, create_steel
from scite.mkappa.mkappa import MKappaAnalysis

print("Imports OK")


## Shared Helper: Convergence Report

`MKappaAnalysis.plot_summary()` is a model-level method that renders the standard
4-panel figure (cross-section | M-κ | strain profile | stress profile).  
The only local helper below is `print_convergence_report` for the numeric summary.


In [ ]:
def print_convergence_report(mk: MKappaAnalysis):
    """Print a compact convergence report for a solved MKappaAnalysis."""
    if mk.M_values is None or len(mk.M_values) == 0:
        print("No data.")
        return
    n_pts = len(mk.M_values)
    N_abs = np.abs(mk.N_values)
    idx_peak = int(np.argmax(mk.M_values))
    print(f"  Points solved     : {n_pts} / {mk.n_kappa}")
    print(f"  M_peak            : {mk.M_values[idx_peak]:.2f} kN·m")
    print(f"  κ at peak         : {mk.kappa_values[idx_peak]*1000:.4f} ‰/mm")
    print(f"  |N| max residual  : {N_abs.max():.4f} kN  (tol = {mk.N_tol:.4f} kN)")
    print(f"  |N| mean residual : {N_abs.mean():.4f} kN")
    print(f"  ε_bot at peak     : {mk.eps_bot_values[idx_peak]:.5f}")

print("Helper defined.")


---
## 1. Rectangular Cross-Section

**Geometry:** b = 300 mm, h = 500 mm  
**Concrete:** C30/37 — f_ck = 30 MPa, α_cc = 0.85, γ_c = 1.5  
**Reinforcement:** 
- Bottom tension layer: 4Ø20 → A_s = 1256.6 mm², z = 50 mm (50 mm cover from bottom)
- Top compression layer: 2Ø16 → A_s = 402.1 mm², z = 450 mm (50 mm cover from top)

**Expected:** M_Rd ≈ 150–180 kN·m for this reinforcement ratio (ρ ≈ 0.84%)


In [ ]:
# ── Rectangular section: geometry ──────────────────────────────────────────
shape_rect = RectangularShape(b=300.0, h=500.0)

# ── Concrete C30/37 (design values, ULS) ───────────────────────────────────
concrete_C30 = EC2ParabolaRectangle(f_ck=30.0, alpha_cc=0.85, gamma_c=1.5)
print(f"f_cd  = {concrete_C30.f_cd:.2f} MPa")
print(f"ε_c2  = {concrete_C30.eps_c2_computed:.5f}")
print(f"ε_cu2 = {concrete_C30.eps_cu2_computed:.5f}")

# ── Steel B500B ─────────────────────────────────────────────────────────────
steel_B500B = create_steel('B500B')
print(f"\nf_yd  = {steel_B500B.f_yd:.2f} MPa")
print(f"ε_yd  = {steel_B500B.eps_yd:.5f}")
print(f"ε_ud  = {steel_B500B.eps_ud:.5f}")

# ── Reinforcement layers ────────────────────────────────────────────────────
# z = distance from bottom fibre
layer_bot_rect = ReinforcementLayer(z=50.0,  A_s=1256.6, material=steel_B500B,
                                    name="4Ø20 bottom (tension)")
layer_top_rect = ReinforcementLayer(z=450.0, A_s=402.1,  material=steel_B500B,
                                    name="2Ø16 top (compression)")
rein_rect = ReinforcementLayout(layers=[layer_bot_rect, layer_top_rect])

print(f"\nρ = {rein_rect.total_area / shape_rect.area * 100:.3f} %")

# ── Assemble cross-section ──────────────────────────────────────────────────
cs_rect = CrossSection(
    shape=shape_rect,
    concrete=concrete_C30,
    reinforcement=rein_rect,
    n_points=200,
)

print(f"\nCross-section properties:")
print(f"  Shape    : {type(cs_rect.shape).__name__}, b={shape_rect.b:.0f} mm, h={shape_rect.h:.0f} mm")
print(f"  A_c      : {cs_rect.A_c:,.0f} mm²")
print(f"  A_s      : {cs_rect.A_s:.1f} mm²")
print(f"  ρ        : {cs_rect.reinforcement_ratio*100:.3f} %")
print(f"  n_layers : {cs_rect.reinforcement.n_layers}")


In [ ]:
# ── Run M-κ analysis (rectangular section) ─────────────────────────────────
mk_rect = MKappaAnalysis(cs=cs_rect, n_kappa=150, N_Ed=0.0)
mk_rect.solve()

print("Rectangular section — convergence report:")
print_convergence_report(mk_rect)


In [ ]:
mk_rect.plot_summary("Rectangular RC Section — b=300 mm, h=500 mm, C30/37, B500B")


### 1.1 Parametric Study: Effect of Reinforcement Ratio (Rectangular Section)

Vary the bottom reinforcement area from under-reinforced to over-reinforced and
compare M-κ curves. All other parameters fixed (b=300, h=500, C30, B500B).


In [ ]:
b, h = 300.0, 500.0
A_s_values = [300.0, 600.0, 1000.0, 1500.0, 2500.0, 4000.0]  # mm² — under → over-reinforced

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(A_s_values)))

for A_s, color in zip(A_s_values, colors):
    layer = ReinforcementLayer(z=50.0, A_s=A_s, material=create_steel('B500B'))
    cs = CrossSection(
        shape=RectangularShape(b=b, h=h),
        concrete=EC2ParabolaRectangle(f_ck=30.0, alpha_cc=0.85, gamma_c=1.5),
        reinforcement=ReinforcementLayout(layers=[layer]),
        n_points=200,
    )
    mk = MKappaAnalysis(cs=cs, n_kappa=120, N_Ed=0.0)
    mk.solve()

    if mk.M_values is not None and len(mk.M_values) > 0:
        rho = A_s / (b * (h - 50))  # effective reinforcement ratio
        ax.plot(mk.kappa_values * 1000.0, mk.M_values,
                color=color, linewidth=2.0,
                label=f"A_s={A_s:.0f} mm² (ρ={rho*100:.2f}%)")

ax.set_xlabel("κ [1/m]", fontsize=12)
ax.set_ylabel("M [kN·m]", fontsize=12)
ax.set_title("M-κ — Effect of Reinforcement Ratio\nb=300 mm, h=500 mm, C30/37, B500B",
             fontsize=12)
ax.legend(fontsize=9, loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 1.2 Parametric Study: Effect of Axial Force N_Ed (Rectangular Section)

Combine bending with axial compression or tension. Useful for columns and
pre-compressed elements.


In [ ]:
N_Ed_values = [-500.0, -200.0, 0.0, 200.0, 500.0]  # kN (neg = compression, pos = tension)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.coolwarm(np.linspace(0.0, 1.0, len(N_Ed_values)))

for N_Ed, color in zip(N_Ed_values, colors):
    mk = MKappaAnalysis(cs=cs_rect, n_kappa=120, N_Ed=N_Ed)
    mk.solve()

    if mk.M_values is not None and len(mk.M_values) > 0:
        label = f"N_Ed = {N_Ed:+.0f} kN ({'compression' if N_Ed < 0 else 'tension' if N_Ed > 0 else 'pure bending'})"
        ax.plot(mk.kappa_values * 1000.0, mk.M_values, color=color, linewidth=2.0, label=label)

ax.set_xlabel("κ [1/m]", fontsize=12)
ax.set_ylabel("M [kN·m]", fontsize=12)
ax.set_title("M-κ — Effect of Axial Force\nb=300 mm, h=500 mm, C30/37, B500B, A_s=1257 mm²",
             fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 2. T-Shaped Cross-Section

T-sections are the standard form for cast-in-place RC floor systems (ribs + slab).
The compression zone shifts into the wide flange — yielding higher moment capacity
and better material utilisation than a rectangular web section alone.

**Geometry:**  
- Web: b_w = 200 mm, h_w = 400 mm  
- Flange (top): b_f = 600 mm, h_f = 150 mm  
- Total height: h_total = h_w + h_f = 550 mm

**Concrete:** C30/37  
**Reinforcement:**  
- Bottom tension layer: 4Ø25 → A_s = 1963 mm², z = 50 mm from bottom  
- (No compression steel — neutral axis in flange at ULS for normal ρ)

**Expected behaviour:**  
- While the N.A. is in the flange the M-κ response resembles a wide rectangular section  
- After the N.A. enters the web, curvature stiffness drops more steeply  
- Peak M typically much higher than equivalent rectangular web section


In [ ]:
# ── T-section geometry ──────────────────────────────────────────────────────
# h_w = web height (bottom part), h_f = flange height (top part)
# h_total = h_w + h_f
shape_T = TShape(b_w=200.0, h_w=400.0, b_f=600.0, h_f=150.0)
print(f"h_total = {shape_T.h_total:.0f} mm")
print(f"A_c     = {shape_T.area:.0f} mm²")
print(f"centroid from bottom = {shape_T.centroid_y:.1f} mm")

# ── Reinforcement ────────────────────────────────────────────────────────────
# 4Ø25 in the web bottom (z = 50 mm from bottom, tension zone)
A_s_T = 4 * np.pi * 12.5**2   # 4Ø25 = 1963.5 mm²
layer_bot_T = ReinforcementLayer(
    z=50.0, A_s=A_s_T, material=create_steel('B500B'), name="4Ø25 bottom"
)
rein_T = ReinforcementLayout(layers=[layer_bot_T])

# ── Assemble cross-section ───────────────────────────────────────────────────
cs_T = CrossSection(
    shape=shape_T,
    concrete=EC2ParabolaRectangle(f_ck=30.0, alpha_cc=0.85, gamma_c=1.5),
    reinforcement=rein_T,
    n_points=200,
)

print(f"\nρ_eff   = {A_s_T / (shape_T.b_w * (shape_T.h_total - 50)) * 100:.3f} %  (w.r.t. web)")
print(f"ρ_gross = {A_s_T / shape_T.area * 100:.3f} %  (w.r.t. total area)")
print(f"\nCross-section properties:")
print(f"  Shape    : {type(cs_T.shape).__name__}")
print(f"  h_total  : {cs_T.h_total:.0f} mm")
print(f"  A_c      : {cs_T.A_c:,.0f} mm²")
print(f"  A_s      : {cs_T.A_s:.1f} mm²")
print(f"  ρ        : {cs_T.reinforcement_ratio*100:.3f} %")
print(f"  n_layers : {cs_T.reinforcement.n_layers}")


In [ ]:
# ── Run M-κ analysis (T-section) ───────────────────────────────────────────
mk_T = MKappaAnalysis(cs=cs_T, n_kappa=150, N_Ed=0.0)
mk_T.solve()

print("T-section — convergence report:")
print_convergence_report(mk_T)


In [ ]:
mk_T.plot_summary("T-Section — b_w=200, h_w=400, b_f=600, h_f=150 mm, C30/37, B500B")


### 2.1 T-Section vs Equivalent Rectangle — Direct Comparison

Compare the T-section with an equivalent rectangular section having the same web
dimensions. Demonstrates the capacity gain from the compression flange.


In [ ]:
# Equivalent rectangular web-only section (b_w=200, h=550=h_total of T-section)
cs_rect_web = CrossSection(
    shape=RectangularShape(b=200.0, h=550.0),
    concrete=EC2ParabolaRectangle(f_ck=30.0, alpha_cc=0.85, gamma_c=1.5),
    reinforcement=ReinforcementLayout(
        layers=[ReinforcementLayer(z=50.0, A_s=A_s_T, material=create_steel('B500B'))]
    ),
    n_points=200,
)
mk_rect_web = MKappaAnalysis(cs=cs_rect_web, n_kappa=150, N_Ed=0.0)
mk_rect_web.solve()

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 6))

if mk_T.M_values is not None and len(mk_T.M_values) > 0:
    ax.plot(mk_T.kappa_values * 1000.0, mk_T.M_values,
            'b-', linewidth=2.5, label="T-section (b_f=600 mm)")

if mk_rect_web.M_values is not None and len(mk_rect_web.M_values) > 0:
    ax.plot(mk_rect_web.kappa_values * 1000.0, mk_rect_web.M_values,
            'r--', linewidth=2.0, label="Rectangle (b_w=200 mm, h=550 mm)")

# Annotate the flange contribution
if (mk_T.M_values is not None and mk_rect_web.M_values is not None
        and len(mk_T.M_values) > 0 and len(mk_rect_web.M_values) > 0):
    M_peak_T = mk_T.M_values.max()
    M_peak_rect = mk_rect_web.M_values.max()
    ax.annotate(
        f"Flange benefit:\n+{M_peak_T - M_peak_rect:.1f} kN·m ({(M_peak_T/M_peak_rect - 1)*100:.0f}%)",
        xy=(mk_T.kappa_values[mk_T.M_values.argmax()] * 1000, M_peak_T),
        xytext=(mk_T.kappa_values[mk_T.M_values.argmax()] * 1000 * 0.5, M_peak_T * 0.85),
        arrowprops=dict(arrowstyle="->", color="navy"),
        color="navy", fontsize=10,
    )

ax.set_xlabel("κ [1/m]", fontsize=12)
ax.set_ylabel("M [kN·m]", fontsize=12)
ax.set_title("T-section vs Rectangular Web — M-κ Comparison\nC30/37, B500B, 4Ø25 at z=50 mm",
             fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 2.2 T-Section: Effect of Flange Width

Demonstrate how flange width influences moment capacity and ductility.


In [ ]:
b_f_values = [200.0, 300.0, 400.0, 600.0, 900.0]  # mm (b_f=b_w=200 is degenerate rectangle)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.plasma(np.linspace(0.1, 0.85, len(b_f_values)))

for b_f, color in zip(b_f_values, colors):
    shape = TShape(b_w=200.0, h_w=400.0, b_f=b_f, h_f=150.0)
    cs = CrossSection(
        shape=shape,
        concrete=EC2ParabolaRectangle(f_ck=30.0, alpha_cc=0.85, gamma_c=1.5),
        reinforcement=ReinforcementLayout(
            layers=[ReinforcementLayer(z=50.0, A_s=A_s_T, material=create_steel('B500B'))]
        ),
        n_points=200,
    )
    mk = MKappaAnalysis(cs=cs, n_kappa=130, N_Ed=0.0)
    mk.solve()

    if mk.M_values is not None and len(mk.M_values) > 0:
        label_str = f"b_f={b_f:.0f} mm" + (" (= web, rect.)" if b_f == 200.0 else "")
        ax.plot(mk.kappa_values * 1000.0, mk.M_values,
                color=color, linewidth=2.0, label=label_str)

ax.set_xlabel("κ [1/m]", fontsize=12)
ax.set_ylabel("M [kN·m]", fontsize=12)
ax.set_title("M-κ — Effect of Flange Width (T-section)\nb_w=200, h_w=400, h_f=150 mm, C30/37, 4Ø25",
             fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 3. Shape Comparison — Rectangular vs T-Section (Normalised)

Normalise M and κ by their peak values to compare the *ductility shape* of the
M-κ response, independent of absolute capacity.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cases = [
    (mk_rect, "Rectangular b=300, h=500", "b-"),
    (mk_T,    "T-section b_w=200, h_w=400, b_f=600, h_f=150", "r-"),
]

ax_abs, ax_norm = axes

for mk, label, style in cases:
    if mk.M_values is None or len(mk.M_values) == 0:
        continue
    kappa_pm = mk.kappa_values * 1000.0

    # Absolute
    ax_abs.plot(kappa_pm, mk.M_values, style, linewidth=2.5, label=label)

    # Normalised
    M_peak = mk.M_values.max()
    kappa_peak = kappa_pm[mk.M_values.argmax()]
    ax_norm.plot(kappa_pm / kappa_peak, mk.M_values / M_peak,
                 style, linewidth=2.5,
                 label=f"{label}\n(M_peak={M_peak:.0f} kN·m, κ_peak={kappa_peak:.3f} 1/m)")

for ax, title in [(ax_abs, "Absolute M-κ"), (ax_norm, "Normalised M/M_peak vs κ/κ_peak")]:
    ax.set_xlabel("κ [1/m]" if ax is ax_abs else "κ / κ_peak [-]", fontsize=12)
    ax.set_ylabel("M [kN·m]" if ax is ax_abs else "M / M_peak [-]", fontsize=12)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("Rectangular vs T-section — C30/37, B500B", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


---
## 4. Solver Robustness Checks

Verify that `MKappaAnalysis` handles edge cases and boundary conditions correctly.

- Very lightly reinforced (near-zero steel) — solver should still converge
- Very heavily reinforced — over-reinforced, compression failure before steel yields
- Varying concrete strength (C20 to C60)


In [ ]:
print("=== Robustness Checks ===\n")

robustness_cases = [
    ("Very low ρ  (A_s=50 mm²)",     dict(A_s=50.0,    f_ck=30.0)),
    ("Low ρ       (A_s=200 mm²)",    dict(A_s=200.0,   f_ck=30.0)),
    ("Medium ρ    (A_s=1257 mm²)",   dict(A_s=1257.0,  f_ck=30.0)),
    ("High ρ      (A_s=3000 mm²)",   dict(A_s=3000.0,  f_ck=30.0)),
    ("Very high ρ (A_s=6000 mm²)",   dict(A_s=6000.0,  f_ck=30.0)),
    ("C20 concrete (A_s=1257 mm²)",  dict(A_s=1257.0,  f_ck=20.0)),
    ("C45 concrete (A_s=1257 mm²)",  dict(A_s=1257.0,  f_ck=45.0)),
    ("C60 concrete (A_s=1257 mm²)",  dict(A_s=1257.0,  f_ck=60.0)),
]

fig, ax = plt.subplots(figsize=(12, 7))
colors = plt.cm.tab10(np.linspace(0, 1, len(robustness_cases)))
results_table = []

for (label, params), color in zip(robustness_cases, colors):
    layer = ReinforcementLayer(z=50.0, A_s=params['A_s'], material=create_steel('B500B'))
    cs = CrossSection(
        shape=RectangularShape(b=300.0, h=500.0),
        concrete=EC2ParabolaRectangle(f_ck=params['f_ck'], alpha_cc=0.85, gamma_c=1.5),
        reinforcement=ReinforcementLayout(layers=[layer]),
        n_points=200,
    )
    mk = MKappaAnalysis(cs=cs, n_kappa=120, N_Ed=0.0)
    try:
        mk.solve()
        n_pts = len(mk.M_values) if mk.M_values is not None else 0
        M_peak = mk.M_values.max() if n_pts > 0 else float('nan')
        N_max = np.abs(mk.N_values).max() if n_pts > 0 else float('nan')
        status = "OK" if n_pts >= 5 else "WARN"
        results_table.append((label, n_pts, M_peak, N_max, status))
        if n_pts > 0:
            ax.plot(mk.kappa_values * 1000.0, mk.M_values,
                    color=color, linewidth=1.8, label=f"{label} | M_pk={M_peak:.0f}")
    except Exception as e:
        results_table.append((label, 0, float('nan'), float('nan'), f"ERROR: {e}"))

# Print table
print(f"{'Case':<40} {'Pts':>5} {'M_peak':>10} {'|N|_max':>10} {'Status':>8}")
print("-" * 78)
for row in results_table:
    print(f"{row[0]:<40} {row[1]:>5} {row[2]:>10.1f} {row[3]:>10.4f} {row[4]:>8}")

ax.set_xlabel("κ [1/m]", fontsize=11)
ax.set_ylabel("M [kN·m]", fontsize=11)
ax.set_title("Robustness — M-κ across reinforcement ratios and concrete grades\nb=300, h=500 mm",
             fontsize=11)
ax.legend(fontsize=7, loc="upper right", ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 5. T-Section Robustness — Varying Flange Geometry and Reinforcement

Stress-test the solver on T-shapes with extreme flange proportions and multiple reinforcement configurations.


In [ ]:
t_cases = [
    ("T: thin flange  h_f=50",   dict(b_w=200, h_w=400, b_f=600, h_f=50,  A_s=1963)),
    ("T: medium flange h_f=150", dict(b_w=200, h_w=400, b_f=600, h_f=150, A_s=1963)),
    ("T: thick flange  h_f=300", dict(b_w=200, h_w=400, b_f=600, h_f=300, A_s=1963)),
    ("T: wide flange   b_f=1200",dict(b_w=200, h_w=400, b_f=1200,h_f=150, A_s=1963)),
    ("T: 2-layer rein  A_s=3000",dict(b_w=200, h_w=400, b_f=600, h_f=150, A_s=3000)),
]

fig, ax = plt.subplots(figsize=(12, 7))
colors = plt.cm.Set2(np.linspace(0, 1, len(t_cases)))

print(f"{'Case':<35} {'Pts':>5} {'M_peak':>10} {'|N|_max':>10}")
print("-" * 65)

for (label, p), color in zip(t_cases, colors):
    cs = CrossSection(
        shape=TShape(b_w=float(p['b_w']), h_w=float(p['h_w']),
                     b_f=float(p['b_f']), h_f=float(p['h_f'])),
        concrete=EC2ParabolaRectangle(f_ck=30.0, alpha_cc=0.85, gamma_c=1.5),
        reinforcement=ReinforcementLayout(
            layers=[ReinforcementLayer(z=50.0, A_s=float(p['A_s']),
                                       material=create_steel('B500B'))]
        ),
        n_points=200,
    )
    mk = MKappaAnalysis(cs=cs, n_kappa=130, N_Ed=0.0)
    try:
        mk.solve()
        n_pts = len(mk.M_values) if mk.M_values is not None else 0
        M_peak = mk.M_values.max() if n_pts > 0 else float('nan')
        N_max = np.abs(mk.N_values).max() if n_pts > 0 else float('nan')
        print(f"{label:<35} {n_pts:>5} {M_peak:>10.1f} {N_max:>10.4f}")
        if n_pts > 0:
            ax.plot(mk.kappa_values * 1000.0, mk.M_values,
                    color=color, linewidth=2.0,
                    label=f"{label}  M_pk={M_peak:.0f}")
    except Exception as e:
        print(f"{label:<35}  ERROR: {e}")

ax.set_xlabel("κ [1/m]", fontsize=11)
ax.set_ylabel("M [kN·m]", fontsize=11)
ax.set_title("T-section robustness — geometry and reinforcement variations\nC30/37, B500B",
             fontsize=11)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
